# Chapter 4: Unicode Text Versus Bytes
*Fluent Python, 2nd Edition -- Luciano Ramalho*

> "Humans use text. Computers speak bytes." -- Esther Nam & Travis Fischer

---

## TL;DR

- **`str`** holds Unicode text (code points); **`bytes`** / **`bytearray`** hold raw octets.
- **Encoding** = str -> bytes; **Decoding** = bytes -> str. Always specify the codec (usually UTF-8).
- The **Unicode sandwich**: decode on input, work with `str`, encode on output. Always pass `encoding=` to `open()`.
- Use **`unicodedata.normalize()`** (NFC/NFD) before comparing strings -- visually identical strings can differ in code points.
- **`str.casefold()`** is the correct way to do case-insensitive comparison (not `.lower()`).
- Use **`locale.strxfrm`** or the **pyuca** library for proper Unicode sorting.
- The `re` and `os` modules are **dual-mode**: they behave differently with `str` vs `bytes` arguments.

**Concepts:** [[Characters vs Bytes]] | [[Encoding and Decoding]] | [[Text File Handling]] | [[Unicode Normalization]] | [[Case Folding and Text Utilities]] | [[Sorting Unicode]] | [[Unicode Database and Dual-Mode APIs]]

---
## 1. Characters vs Bytes

A **character** is an abstract identity (a Unicode code point, e.g. U+0041 = 'A').
A **byte** is a concrete 8-bit value (0-255).

The same character can be represented by *different* byte sequences depending on the **encoding**.

See also: [[Characters vs Bytes]]

In [ ]:
# The fundamental distinction: str vs bytes
s = 'cafe\u0301'  # 'e' + combining acute accent
print(f"str:  {s!r}  len={len(s)}")

b_utf8 = s.encode('utf-8')
print(f"bytes (UTF-8): {b_utf8!r}  len={len(b_utf8)}")

b_utf16 = s.encode('utf-16')
print(f"bytes (UTF-16): {b_utf16!r}  len={len(b_utf16)}")

In [ ]:
# bytes vs bytearray
cafe = bytes('cafe\u0301', encoding='utf_8')
print(f"bytes: {cafe!r}")
print(f"cafe[0] = {cafe[0]}  (int, not character!)")
print(f"cafe[:1] = {cafe[:1]!r}  (slice -> bytes)")

# bytearray is mutable
cafe_arr = bytearray(cafe)
cafe_arr[0] = 0x43  # Change 'c' to 'C'
print(f"mutated bytearray: {cafe_arr!r}")

In [ ]:
# Key insight: indexing bytes gives int, slicing gives bytes
data = b'hello'
print(type(data[0]), data[0])     # <class 'int'> 104
print(type(data[:1]), data[:1])   # <class 'bytes'> b'h'

# Contrast with str where s[0] == s[:1]
text = 'hello'
print(text[0] == text[:1])  # True for str, unlike bytes

---
## 2. Encoding and Decoding

**Encoding** = str -> bytes (human text -> machine storage)
**Decoding** = bytes -> str (machine storage -> human text)

Python bundles 100+ codecs. The most important: UTF-8 (web standard, 97% of sites).

See also: [[Encoding and Decoding]]

In [ ]:
# Encoding: str -> bytes
word = 'café'  # pre-composed e-acute

# Same string, different byte representations
for codec in ['latin_1', 'utf_8', 'utf_16']:
    encoded = word.encode(codec)
    print(f"{codec:10s} -> {encoded!r:30s}  ({len(encoded)} bytes)")

In [ ]:
# Handling encode errors
city = 'S\u00e3o Paulo'  # Sao Paulo with tilde

# UTF-8 handles everything
print("utf_8:", city.encode('utf_8'))

# ASCII can't handle non-ASCII characters
try:
    city.encode('ascii')
except UnicodeEncodeError as e:
    print(f"UnicodeEncodeError: {e}")

# Error handling strategies
print("ignore: ", city.encode('ascii', errors='ignore'))
print("replace:", city.encode('ascii', errors='replace'))
print("xmlref: ", city.encode('ascii', errors='xmlcharrefreplace'))

In [ ]:
# Handling decode errors
octets = b'Montr\xe9al'  # 'Montreal' encoded in latin-1

print("cp1252:", octets.decode('cp1252'))    # Correct: Montr\xe9al
print("iso8859_7:", octets.decode('iso8859_7'))  # Greek codec: wrong char

try:
    octets.decode('utf_8')
except UnicodeDecodeError as e:
    print(f"UnicodeDecodeError: {e}")

# The 'replace' handler shows where problems are
print("utf_8 replace:", octets.decode('utf_8', errors='replace'))

In [ ]:
# BOM (Byte Order Mark) in UTF-16
text = 'AB'
u16 = text.encode('utf_16')
print(f"utf_16:   {list(u16)}  (starts with BOM: 255, 254)")

u16le = text.encode('utf_16le')
print(f"utf_16le: {list(u16le)}  (no BOM, explicit little-endian)")

u16be = text.encode('utf_16be')
print(f"utf_16be: {list(u16be)}  (no BOM, explicit big-endian)")

# UTF-8 BOM (rare but exists, especially from Windows apps)
u8sig = text.encode('utf-8-sig')
print(f"utf-8-sig: {u8sig!r}  (BOM = EF BB BF)")

---
## 3. Text File Handling -- The Unicode Sandwich

The **Unicode sandwich** pattern:
1. **Decode** bytes to str as early as possible (on input)
2. Work exclusively with **str** in business logic
3. **Encode** str to bytes as late as possible (on output)

**Critical rule:** Always pass `encoding=` when calling `open()`.

See also: [[Text File Handling]]

In [ ]:
import os, tempfile

# Demonstrate the encoding pitfall
tmpdir = tempfile.mkdtemp()
filepath = os.path.join(tmpdir, 'cafe.txt')

# Write with explicit encoding
with open(filepath, 'w', encoding='utf_8') as f:
    chars_written = f.write('cafe\u0301')
    print(f"Characters written: {chars_written}")

# Check actual file size in bytes
print(f"File size in bytes: {os.stat(filepath).st_size}")

# Read with correct encoding
with open(filepath, 'r', encoding='utf_8') as f:
    print(f"Read back: {f.read()!r}")

# Read as raw bytes to see what's on disk
with open(filepath, 'rb') as f:
    print(f"Raw bytes: {f.read()!r}")

# Clean up
os.remove(filepath)
os.rmdir(tmpdir)

In [ ]:
# Encoding defaults vary by platform!
import locale, sys

print(f"locale.getpreferredencoding(): {locale.getpreferredencoding()!r}")
print(f"sys.getdefaultencoding():      {sys.getdefaultencoding()!r}")
print(f"sys.getfilesystemencoding():   {sys.getfilesystemencoding()!r}")
print(f"sys.stdout.encoding:           {sys.stdout.encoding!r}")

# On macOS/Linux these are all UTF-8.
# On Windows, locale.getpreferredencoding() may be 'cp1252'
# -- which is why you should ALWAYS specify encoding= explicitly.

---
## 4. Unicode Normalization

The same visual character can have multiple code point representations.
`unicodedata.normalize()` resolves this ambiguity.

| Form | Effect | Use Case |
|------|--------|----------|
| **NFC** | Compose (shortest) | Default for storage, W3C recommended |
| **NFD** | Decompose (base + combining) | Internal processing |
| **NFKC** | Compatibility compose | Search/indexing (lossy!) |
| **NFKD** | Compatibility decompose | Search/indexing (lossy!) |

See also: [[Unicode Normalization]]

In [ ]:
from unicodedata import normalize, name

# The classic problem: two ways to write 'e with acute'
s1 = 'caf\u00e9'                         # e-acute as single code point
s2 = 'cafe\N{COMBINING ACUTE ACCENT}'    # e + combining accent

print(f"s1 = {s1!r}  len={len(s1)}")
print(f"s2 = {s2!r}  len={len(s2)}")
print(f"s1 == s2: {s1 == s2}")  # False! Different code points
print(f"Look identical: '{s1}' vs '{s2}'")
print()

# NFC: compose to shortest form
nfc1 = normalize('NFC', s1)
nfc2 = normalize('NFC', s2)
print(f"NFC(s1) len={len(nfc1)}, NFC(s2) len={len(nfc2)}")
print(f"NFC equal: {nfc1 == nfc2}")  # True!

# NFD: decompose to base + combining marks
nfd1 = normalize('NFD', s1)
nfd2 = normalize('NFD', s2)
print(f"NFD(s1) len={len(nfd1)}, NFD(s2) len={len(nfd2)}")
print(f"NFD equal: {nfd1 == nfd2}")  # True!

In [ ]:
from unicodedata import normalize, name

# NFC can also normalize single characters
ohm = '\u2126'
omega = normalize('NFC', ohm)
print(f"ohm:   U+{ord(ohm):04X} {name(ohm)}")
print(f"omega: U+{ord(omega):04X} {name(omega)}")
print(f"ohm == omega: {ohm == omega}")
print(f"NFC(ohm) == NFC(omega): {normalize('NFC', ohm) == normalize('NFC', omega)}")

In [ ]:
from unicodedata import normalize, name

# NFKC/NFKD: compatibility normalization (lossy!)
half = '\N{VULGAR FRACTION ONE HALF}'
print(f"Original: {half!r} -> '{half}'")
nfkc_half = normalize('NFKC', half)
print(f"NFKC:     {nfkc_half!r}")
for ch in nfkc_half:
    print(f"  U+{ord(ch):04X} {name(ch)}")

print()

# Danger: NFKC can change meaning!
four_sq = '4\u00b2'  # 4 squared
print(f"Original: {four_sq!r} -> '{four_sq}'")
print(f"NFKC:     {normalize('NFKC', four_sq)!r}  -- meaning changed!")

print()

# Micro sign -> Greek mu
micro = '\u00b5'
mu = normalize('NFKC', micro)
print(f"micro: U+{ord(micro):04X} {name(micro)}")
print(f"NFKC:  U+{ord(mu):04X} {name(mu)}")

---
## 5. Case Folding and Text Utilities

`str.casefold()` is more aggressive than `str.lower()` -- it handles ~300 extra code points.

See also: [[Case Folding and Text Utilities]]

In [ ]:
from unicodedata import normalize, name

# casefold() vs lower() -- usually the same, but not always
micro = '\u00b5'  # MICRO SIGN
print(f"micro.lower():    {micro.lower()!r}  ({name(micro.lower())})")
print(f"micro.casefold(): {micro.casefold()!r}  ({name(micro.casefold())})")

eszett = '\u00df'  # LATIN SMALL LETTER SHARP S
print(f"eszett.lower():    {eszett.lower()!r}")
print(f"eszett.casefold(): {eszett.casefold()!r}")  # -> 'ss' !

In [ ]:
from unicodedata import normalize

# Utility functions for safe text comparison
def nfc_equal(str1, str2):
    """Case-sensitive comparison using NFC normalization."""
    return normalize('NFC', str1) == normalize('NFC', str2)

def fold_equal(str1, str2):
    """Case-insensitive comparison using NFC + casefold."""
    return (normalize('NFC', str1).casefold() ==
            normalize('NFC', str2).casefold())

# Demo
s1 = 'caf\u00e9'
s2 = 'cafe\u0301'
print(f"s1 == s2:         {s1 == s2}")
print(f"nfc_equal(s1,s2): {nfc_equal(s1, s2)}")

s3 = 'Stra\u00dfe'   # Strasse with eszett
s4 = 'strasse'
print(f"nfc_equal(s3,s4):  {nfc_equal(s3, s4)}")
print(f"fold_equal(s3,s4): {fold_equal(s3, s4)}")

In [ ]:
import unicodedata

# Removing diacritics (accent stripping)
def shave_marks(txt):
    """Remove all diacritic marks."""
    norm_txt = unicodedata.normalize('NFD', txt)
    shaved = ''.join(c for c in norm_txt
                     if not unicodedata.combining(c))
    return unicodedata.normalize('NFC', shaved)

# Demo
samples = ['caf\u00e9', 'a\u00e7a\u00ed', '\u0396\u03ad\u03c6\u03c5\u03c1\u03bf\u03c2']
for s in samples:
    print(f"{s:15s} -> {shave_marks(s)}")

---
## 6. Sorting Unicode Text

`sorted()` compares code points, not linguistic order. This fails for accented characters.

Solutions:
1. `locale.strxfrm` -- OS-dependent, requires locale setup
2. `pyuca` library -- portable Unicode Collation Algorithm

See also: [[Sorting Unicode]]

In [ ]:
# The problem: sorted() uses code point order
fruits = ['caju', 'atemoia', 'caj\u00e1', 'a\u00e7a\u00ed', 'acerola']
print("Default sorted:", sorted(fruits))
# 'acai' and 'caja' are in wrong positions because
# accented chars have higher code points

# Expected linguistic order:
# ['acai', 'acerola', 'atemoia', 'caja', 'caju']

In [ ]:
# Solution 1: locale.strxfrm (OS-dependent)
import locale

# Try to set a locale -- may not work on all systems
try:
    locale.setlocale(locale.LC_COLLATE, 'en_US.UTF-8')
    fruits = ['caju', 'atemoia', 'caj\u00e1', 'a\u00e7a\u00ed', 'acerola']
    sorted_fruits = sorted(fruits, key=locale.strxfrm)
    print("locale sorted:", sorted_fruits)
except locale.Error as e:
    print(f"Locale not available: {e}")
    # Fallback: strip accents then sort
    import unicodedata
    def sort_key(s):
        return unicodedata.normalize('NFD', s).encode('ascii', 'ignore').decode()
    print("fallback sorted:", sorted(fruits, key=sort_key))

In [ ]:
# Solution 2: pyuca (portable, recommended)
# Install: pip install pyuca
try:
    import pyuca
    coll = pyuca.Collator()
    fruits = ['caju', 'atemoia', 'caj\u00e1', 'a\u00e7a\u00ed', 'acerola']
    print("pyuca sorted:", sorted(fruits, key=coll.sort_key))
except ImportError:
    print("pyuca not installed. Install with: pip install pyuca")
    # Demonstrate the concept with a simple fallback
    import unicodedata
    def nfd_sort_key(s):
        return unicodedata.normalize('NFD', s)
    fruits = ['caju', 'atemoia', 'caj\u00e1', 'a\u00e7a\u00ed', 'acerola']
    print("NFD fallback sorted:", sorted(fruits, key=nfd_sort_key))

---
## 7. The Unicode Database and Dual-Mode APIs

The `unicodedata` module gives access to character metadata: names, categories,
numeric values. The `re` and `os` modules behave differently with `str` vs `bytes`.

See also: [[Unicode Database and Dual-Mode APIs]]

In [ ]:
import unicodedata

# Exploring the Unicode database
chars = ['A', '\u00e9', '\u2126', '\u00bd', '\u00b5', '\U0001D11E']
for ch in chars:
    try:
        n = unicodedata.name(ch)
    except ValueError:
        n = '(no name)'
    cat = unicodedata.category(ch)
    print(f"U+{ord(ch):04X}  {ch:2s}  cat={cat}  name={n}")

In [ ]:
import unicodedata

# Numeric meaning of characters
sample = '1\u00bc\u00b2\u0969\u2466'
for char in sample:
    try:
        num_val = unicodedata.numeric(char)
    except ValueError:
        num_val = None
    print(f"U+{ord(char):04x}  {char}  "
          f"isdecimal={char.isdecimal()!s:5s}  "
          f"isdigit={char.isdigit()!s:5s}  "
          f"isnumeric={char.isnumeric()!s:5s}  "
          f"numeric={num_val}")

In [ ]:
# Building a simple character finder (like the book's cf.py)
import unicodedata, sys

def find_chars(*query_words, start=32, end=0x10000):
    """Find Unicode characters whose names contain all query words."""
    query = {w.upper() for w in query_words}
    results = []
    for code in range(start, end):
        char = chr(code)
        char_name = unicodedata.name(char, None)
        if char_name and query.issubset(char_name.split()):
            results.append((code, char, char_name))
    return results

# Find some fun characters
for code, char, char_name in find_chars('cat', 'face')[:5]:
    print(f"U+{code:04X}  {char}  {char_name}")

In [ ]:
import re

# Dual-mode: str vs bytes in regular expressions
text_str = "Ramanujan saw \u0be7\u0bed\u0be8\u0bef as 1729 = 1\u00b3 + 12\u00b3"
text_bytes = text_str.encode('utf_8')

# str patterns match Unicode digits
re_numbers_str = re.compile(r'\d+')
print("str  numbers:", re_numbers_str.findall(text_str))

# bytes patterns match only ASCII digits
re_numbers_bytes = re.compile(rb'\d+')
print("bytes numbers:", re_numbers_bytes.findall(text_bytes))

# str \w matches Unicode letters
re_words_str = re.compile(r'\w+')
print("str  words:", re_words_str.findall(text_str))

# bytes \w matches only ASCII letters+digits
re_words_bytes = re.compile(rb'\w+')
print("bytes words:", re_words_bytes.findall(text_bytes))

In [ ]:
import os, tempfile

# Dual-mode: str vs bytes in os functions
tmpdir = tempfile.mkdtemp()
# Create a file with a Unicode name
filepath = os.path.join(tmpdir, 'digits-of-\u03c0.txt')
with open(filepath, 'w') as f:
    f.write('3.14159')

# os.listdir with str argument -> str results
print("str listdir: ", os.listdir(tmpdir))

# os.listdir with bytes argument -> bytes results
print("bytes listdir:", os.listdir(tmpdir.encode()))

# Clean up
os.remove(filepath)
os.rmdir(tmpdir)

---
## Exercises

Test your understanding of Unicode text handling.

In [ ]:
# Exercise 1: Encoding Detective
# What encoding was used? Figure out the original string.
mystery_bytes = b'caf\xc3\xa9'

# Try decoding with different codecs and find the right one
for codec in ['ascii', 'latin_1', 'utf_8', 'cp1252']:
    try:
        result = mystery_bytes.decode(codec)
        print(f"{codec:10s} -> {result!r}")
    except UnicodeDecodeError as e:
        print(f"{codec:10s} -> ERROR: {e}")

# Which one gives the correct result?

In [ ]:
# Exercise 2: Normalization Challenge
# Are these strings equal? Make them compare as True.
from unicodedata import normalize

a = '\u00f1'          # n with tilde (single code point)
b = 'n\u0303'         # n + combining tilde

print(f"a = {a!r} (len={len(a)})")
print(f"b = {b!r} (len={len(b)})")
print(f"a == b: {a == b}")

# Fix: normalize both, then compare
# YOUR CODE: use normalize() to make them equal
a_norm = normalize('NFC', a)
b_norm = normalize('NFC', b)
print(f"After NFC: {a_norm == b_norm}")  # Should be True

In [ ]:
# Exercise 3: Case-insensitive comparison done right
# Compare these strings case-insensitively using casefold
from unicodedata import normalize

pairs = [
    ('Stra\u00dfe', 'STRASSE'),  # German sharp-s
    ('caf\u00e9', 'CAFE\u0301'),  # composed vs decomposed + case
    ('\u00b5', '\u03bc'),         # micro sign vs Greek mu
]

def robust_equal(s1, s2):
    """Case-insensitive, normalization-aware comparison."""
    return (normalize('NFC', s1).casefold() ==
            normalize('NFC', s2).casefold())

for s1, s2 in pairs:
    print(f"{s1!r:15s} vs {s2!r:15s} -> "
          f"==: {s1==s2!s:5s}  "
          f"robust: {robust_equal(s1, s2)}")

---
## Key Takeaways

1. **`str` is text, `bytes` is data.** Never confuse the two. Encoding bridges them.

2. **Always specify `encoding=`** when opening text files. Never rely on platform defaults.

3. **The Unicode sandwich:** decode early, process as `str`, encode late.

4. **Normalize before comparing.** Use `NFC` for general use; `NFKC`/`NFKD` only for search/indexing (they're lossy).

5. **Use `casefold()`, not `lower()`** for case-insensitive comparisons. It handles edge cases like sharp-s and micro sign.

6. **Sorting Unicode requires extra work.** Use `locale.strxfrm` or `pyuca` -- never rely on default `sorted()` for non-ASCII text.

7. **`re` and `os` are dual-mode.** `bytes` patterns match only ASCII; `str` patterns match full Unicode.

---

*Next chapter: [[05-data-class-builders|Data Class Builders]]*